# 01. Stage 1: No-Pandemic Counterfactual Forecasts

**Geographic vintage:** September 2018 OMB. **Information cutoff:**
February 29, 2020. The last full QCEW quarter published by then is 2019 Q2.
Consequently, **model fitting, tuning, and selection stop at June 2019**.

Each CBSA independently evaluates a prespecified grid of seasonal naive,
seasonal drift, log-linear trend-seasonal, ETS structures, and SARIMA orders.
Three rolling-origin holdouts, ending by June 2019, determine the
lowest mean normalized MAE. The primary analysis uses 12-month folds;
24/36/48-month horizon variants can be rerun **in separate project roots**. The winning specification is refitted through
June 2019, then extrapolated from July 2019 through December 2023. Realized
2020–2023 employment is joined *only afterward* to calculate gaps.

**Caution:** The winning cross-validation score is a selection score and may be
optimistic because the same pre-pandemic folds tune candidate specifications.
July 2019–February 2020 observations form a separate post-selection diagnostic
window and are never used for fitting or selection. Long-horizon forecasts are
extrapolations, not identified causal effects. Downloaded historical QCEW files
may reflect revisions made after February 2020; archived release-vintage data
would be required for an exact real-time reconstruction.
**Corrections:** seasonal drift uses the correct annual seasonal-base offset; log forecasts support mean-bias adjustment; all extrapolations must pass an explicit bound versus the pre-cutoff employment level. Results also carry a flat-seasonal/selected-model ensemble and a damped-selected sensitivity. Set `RL_STAGE1_LONG_CV=1` for **54-month pre-pandemic historical stress tests**. For each historical pseudo-origin, the simple baseline is selected using only observations available by that pseudo-origin, and its forecast errors are used to create three plausible *stress scenarios* for downstream outcomes. These are illustrative **not calibrated confidence intervals**. Set `RL_STAGE1_RETRO_SPEC=1` for an additional *retrospective* check of the 2019 selected model, which necessarily has selection look-ahead relative to the older origins; this is computationally expensive and is diagnostic, not post-2020 tuning. Uncertainty in the estimated counterfactual is not eliminated by these sensitivities.

**Full-run settings in this package:** 12-month selection remains primary; full CBSAs, ETS and SARIMA standard grids, log-bias correction, the 54-month historical stress backtests and the retrospective model-family comparison is disabled. The retrospective comparison uses the 2019-selected specification at earlier origins and has selection look-ahead; it is descriptive only. Distinct 24/36/48-month *selection* experiments must use separate project directories (see `FULL_RUN_GUIDE.md`).


Revised primary selection: December 2015/2016/2017 origins, normalized RMSE then MAE. December 2018 plus 12 months is inadmissible because its validation observations were not fully released by the cutoff. Final training remains June 2019. No February-2020-reference retrospective alternative is run.


In [1]:
from pathlib import Path
import os, json, hashlib, warnings, time, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.linear_model import LinearRegression
from threadpoolctl import threadpool_limits

ROOT = (Path(os.environ['RL_PROJECT_ROOT']).expanduser().resolve()
        if os.environ.get('RL_PROJECT_ROOT') else next(
            (p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'data'/'processed'/'omb2018_geography_manifest.json').is_file()), None))
if ROOT is None:
    raise FileNotFoundError('Run notebook 00 first (2018 OMB panel and manifest).')
PROC, OUT = ROOT/'data'/'processed', ROOT/'output'
OUT.mkdir(parents=True, exist_ok=True)
GEO = json.loads((PROC/'omb2018_geography_manifest.json').read_text())
if GEO['geography_vintage'] != '2018-09':
    raise ValueError('Expected September 2018 OMB delineations; refuse mixed vintages.')
PANEL = PROC/'qcew_cbsa_monthly_2010_2023_omb2018.csv'

def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as handle:
        for b in iter(lambda: handle.read(4*1024*1024), b''):
            h.update(b)
    return h.hexdigest()

if sha256(PANEL) != GEO['panel_sha256']:
    raise ValueError('2018 panel hash differs from notebook 00 geography manifest.')
TRAIN_END = pd.Timestamp('2019-06-01')
FORECAST_END = pd.Timestamp('2023-12-01')
# The original prespecified design selects on 12 months. Longer-horizon
# variants are explicitly separate scientific analyses, not post-hoc choices.
VALIDATION_MONTHS = int(os.getenv('RL_STAGE1_SELECTION_HORIZON','12'))
ORIGINS_BY_HORIZON={
    12:('2015-12-01','2016-12-01','2017-12-01'),
    24:('2015-06-01','2016-06-01','2017-06-01'),
    36:('2014-06-01','2015-06-01','2016-06-01'),
    48:('2013-06-01','2014-06-01','2015-06-01')}
if VALIDATION_MONTHS not in ORIGINS_BY_HORIZON:
    raise ValueError('RL_STAGE1_SELECTION_HORIZON must be one of 12,24,36,48')
if VALIDATION_MONTHS!=12 and os.getenv('RL_ALLOW_ALTERNATE_SELECTION','0')!='1':
    raise ValueError('Long-horizon selection is a separate experimental design. '
                     'Set RL_ALLOW_ALTERNATE_SELECTION=1 and use a separate RL_PROJECT_ROOT.')
VALIDATION_ORIGINS=tuple(pd.Timestamp(x) for x in ORIGINS_BY_HORIZON[VALIDATION_MONTHS])
RUN_ETS = True  # include ETS family regardless of earlier smoke-test environment
RUN_SARIMA = True  # include SARIMA family regardless of earlier smoke-test environment
ETS_GRID = 'standard'  # all six prespecified ETS candidates
SARIMA_GRID = 'standard'  # all fifteen prespecified SARIMA candidates
N_JOBS = 8  # full national run; conservative for a 32-GB workstation
MAX_CBSAS = None  # intentionally ALL eligible CBSAs; never honor leftover smoke-test variables
SARIMA_MAXITER = int(os.environ.get('RL_SARIMA_MAXITER','80'))
LONG_CV = True  # full: three 54-month pre-pandemic historical backtests and error stress paths
RETRO_SPEC = False  # descriptive hindsight model-family diagnostic (selection look-ahead!)
LOG_BIAS_ADJUST = True  # Duan smearing/lognormal predictive mean adjustment
FORECAST_MAX_LEVEL_MULTIPLE = float(os.getenv('RL_FORECAST_MAX_MULTIPLE','4.0'))
FORECAST_MIN_LEVEL_MULTIPLE = float(os.getenv('RL_FORECAST_MIN_MULTIPLE','0.25'))
if not (0 < FORECAST_MIN_LEVEL_MULTIPLE < 1 < FORECAST_MAX_LEVEL_MULTIPLE):
    raise ValueError('Invalid forecast plausibility multipliers')
if N_JOBS < 1 or (MAX_CBSAS is not None and MAX_CBSAS < 1):
    raise ValueError('RL_N_JOBS and RL_MAX_CBSAS must be positive.')
if ETS_GRID not in ('fast','standard'):
    raise ValueError('RL_ETS_GRID must be fast or standard')
if SARIMA_GRID not in ('smoke','fast','standard','extended'):
    raise ValueError('RL_SARIMA_GRID must be smoke, fast, standard, or extended')
np.random.seed(2026)
print('Project:',ROOT)
print('2018 OMB geography | as-of: 2020-02-29 | latest fitting month:',TRAIN_END.date())
print('ETS:',RUN_ETS,ETS_GRID,'SARIMA:',RUN_SARIMA,SARIMA_GRID,'workers:',N_JOBS)
print('FULL MODE: every CBSA; all standard ETS/SARIMA candidates; 54-month historical diagnostics; retrospective comparison')

Project: c:\Users\adamt\OneDrive\Documents\Directed Practicum\regional-labor-resilience
2018 OMB geography | as-of: 2020-02-29 | latest fitting month: 2019-06-01
ETS: True standard SARIMA: True standard workers: 8
FULL MODE: every CBSA; all standard ETS/SARIMA candidates; 54-month historical diagnostics; retrospective comparison


## 1. Load the verified 2018 OMB panel

Stage 1 outcomes may extend through 2023, but every model-fitting call below
receives **only** the June 2019-and-earlier slice.

In [2]:
panel = pd.read_csv(PANEL, dtype={'cbsa_code':'string'},parse_dates=['date'])
required = {'cbsa_code','date','employment','cbsa_type'}
if not required.issubset(panel):
    raise ValueError(f'Missing panel fields: {sorted(required-set(panel))}')
panel['cbsa_code'] = panel.cbsa_code.astype('string').str.zfill(5)
panel['employment'] = pd.to_numeric(panel.employment,errors='coerce')
if panel.duplicated(['cbsa_code','date']).any():
    raise ValueError('Duplicate CBSA and month')
if panel.employment.isna().any() or (panel.employment <= 0).any():
    raise ValueError('Missing or nonpositive employment in the Stage 1 panel')
if panel.cbsa_code.nunique() != GEO['n_cbsa']:
    raise ValueError('CBSA count changed since notebook 00')
months = pd.date_range('2010-01-01','2023-12-01',freq='MS')
if not panel.groupby('cbsa_code').date.nunique().eq(len(months)).all():
    raise ValueError('Incomplete monthly CBSA panel')
pre = panel.loc[panel.date <= TRAIN_END].copy()
cbsa_ids = sorted(pre.cbsa_code.unique())
if MAX_CBSAS:
    cbsa_ids = cbsa_ids[:MAX_CBSAS]
tasks = [(cbsa, group.sort_values('date').employment.to_numpy(dtype=float))
         for cbsa,group in pre.loc[pre.cbsa_code.isin(cbsa_ids)].groupby('cbsa_code',sort=True)]
expected_pre_dates = pd.date_range('2010-01-01',TRAIN_END,freq='MS')
if any(len(y) != len(expected_pre_dates) for _,y in tasks):
    raise ValueError('Incomplete pre-cutoff employment series')
print('CBSAs:',len(tasks),'out of',GEO['n_cbsa'])

CBSAs: 936 out of 936


## 2. Define candidate forecasting specifications

ETS candidates vary trend,
damping and seasonal form. SARIMA candidates vary both nonseasonal and
seasonal orders, with annual (12-month) seasonality and a log transform.

Choose `RL_SARIMA_GRID=fast` for an initial national-scale run, `standard`
(default) for the primary analysis, or `extended` for a sensitivity analysis.
Larger grids are much slower: approximate SARIMA fits per CBSA =
`number of grid candidates × 3 validation origins + 1 final refit`.

In [3]:
def seasonal_naive(y,h):
    return np.tile(np.asarray(y[-12:],float),int(np.ceil(h/12)))[:h]

def seasonal_drift(y,h):
    y=np.asarray(y,float)
    years=(len(y)-12)/12
    annual_change=(y[-12:].mean()-y[:12].mean())/years
    return seasonal_naive(y,h) + annual_change*np.ceil((np.arange(h)+1)/12)

def trend_seasonal(y,h):
    y=np.asarray(y,float)
    train_t=np.arange(len(y))
    test_t=np.arange(len(y),len(y)+h)
    def design(t):
        # Monthly level (not seasonal-difference) deterministic log trend.
        return np.column_stack([t]+[(t%12==k).astype(float) for k in range(1,12)])
    fit=LinearRegression().fit(design(train_t),np.log(y))
    pred_log=fit.predict(design(test_t))
    if not np.isfinite(pred_log).all():
        raise ValueError('Nonfinite trend-seasonal log forecast')
    # Duan smearing: approximate the conditional employment mean, not just median.
    smearing=np.mean(np.exp(np.log(y)-fit.predict(design(train_t)))) if LOG_BIAS_ADJUST else 1.0
    return np.exp(np.clip(pred_log,-700,700))*smearing

BASELINES = [
    {'family':'seasonal_naive','spec':'seasonal_naive','config':{}},
    {'family':'seasonal_drift','spec':'seasonal_drift','config':{}},
    {'family':'trend_seasonal','spec':'trend_seasonal_log','config':{}},
]

ets_options=[
    {'trend':None,'seasonal':'add','damped_trend':False},
    {'trend':'add','seasonal':'add','damped_trend':False},
    {'trend':'add','seasonal':'add','damped_trend':True},
    {'trend':None,'seasonal':'mul','damped_trend':False},
    {'trend':'add','seasonal':'mul','damped_trend':False},
    {'trend':'add','seasonal':'mul','damped_trend':True},
]
if ETS_GRID=='fast':
    ets_options=ets_options[:3]

sarima_options = [
    ((0,1,1),(0,1,1,12)),
    ((1,1,0),(0,1,1,12)),
    ((1,1,1),(0,1,1,12)),
    ((0,1,1),(1,1,0,12)),
    ((1,1,0),(1,1,0,12)),
    ((1,1,1),(1,1,0,12)),
    ((0,1,1),(1,1,1,12)),
    ((1,1,0),(1,1,1,12)),
    ((1,1,1),(1,1,1,12)),
    ((0,1,1),(0,0,1,12)),
    ((1,1,0),(0,0,1,12)),
    ((1,1,1),(0,0,1,12)),
    ((0,1,1),(1,0,0,12)),
    ((1,1,0),(1,0,0,12)),
    ((1,1,1),(1,0,0,12)),
]
if SARIMA_GRID=='smoke':
    sarima_options=sarima_options[:2]
elif SARIMA_GRID=='fast':
    sarima_options=sarima_options[:6]
elif SARIMA_GRID=='extended':
    sarima_options += [
        ((0,0,1),(0,1,1,12)), ((1,0,0),(0,1,1,12)),
        ((1,0,1),(0,1,1,12)), ((0,0,1),(1,1,0,12)),
        ((1,0,0),(1,1,0,12)), ((1,0,1),(1,1,0,12)),
        ((0,0,1),(0,0,1,12)), ((1,0,0),(0,0,1,12)),
        ((1,0,1),(0,0,1,12)), ((2,1,0),(0,1,1,12)),
        ((0,1,2),(0,1,1,12)), ((2,1,0),(1,1,0,12)),
        ((0,1,2),(1,1,0,12)),
    ]

candidates=list(BASELINES)
if RUN_ETS:
    for conf in ets_options:
        spec=f"ETS_trend-{conf['trend'] or 'none'}_season-{conf['seasonal']}_damped-{int(conf['damped_trend'])}"
        candidates.append({'family':'ets','spec':spec,'config':conf})
if RUN_SARIMA:
    for order,seasonal_order in sarima_options:
        spec=(f"SARIMA_{''.join(map(str,order))}_"
              f"{''.join(map(str,seasonal_order[:3]))}_s12_log")
        candidates.append({'family':'sarima','spec':spec,
                           'config':{'order':list(order),'seasonal_order':list(seasonal_order)}})
assert len(set(x['spec'] for x in candidates)) == len(candidates)
print('Candidate specifications:',len(candidates),'(ETS:',len(ets_options) if RUN_ETS else 0,
      '| SARIMA:',len(sarima_options) if RUN_SARIMA else 0,')')

def fit_forecast(y, h, candidate):
    # Crucial: y is provided as a pre-cutoff slice, not the full panel.
    family=candidate['family']; cfg=candidate['config']
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        if family=='seasonal_naive':
            pred=seasonal_naive(y,h)
        elif family=='seasonal_drift':
            pred=seasonal_drift(y,h)
        elif family=='trend_seasonal':
            pred=trend_seasonal(y,h)
        elif family=='ets':
            from statsmodels.tsa.holtwinters import ExponentialSmoothing
            model=ExponentialSmoothing(
                np.asarray(y,float),trend=cfg['trend'],seasonal=cfg['seasonal'],
                damped_trend=cfg['damped_trend'],seasonal_periods=12,
                initialization_method='estimated')
            fitted=model.fit(optimized=True,use_brute=False)
            pred=np.asarray(fitted.forecast(h),float)
        elif family=='sarima':
            from statsmodels.tsa.statespace.sarimax import SARIMAX
            model=SARIMAX(np.log(np.asarray(y,float)),
                order=tuple(cfg['order']),seasonal_order=tuple(cfg['seasonal_order']),
                trend='n',enforce_stationarity=False,enforce_invertibility=False)
            fit=model.fit(disp=False,maxiter=SARIMA_MAXITER)
            if not fit.mle_retvals.get('converged',False):
                raise ValueError('SARIMA optimizer did not converge')
            distribution=fit.get_forecast(steps=h)
            log_pred=np.asarray(distribution.predicted_mean,float)
            log_var=np.asarray(distribution.var_pred_mean,float)
            if not np.isfinite(log_pred).all():
                raise ValueError('Nonfinite log-SARIMA forecast')
            # Lognormal approximation; compare LOG_BIAS_ADJUST=0 as sensitivity.
            if LOG_BIAS_ADJUST:
                if not np.isfinite(log_var).all() or (log_var<0).any():
                    raise ValueError('Invalid log-SARIMA predictive variance')
                log_pred=log_pred+0.5*log_var
            pred=np.exp(np.clip(log_pred,-700,700))
        else:
            raise ValueError(f'Unknown model family: {family}')
    pred=np.asarray(pred,dtype=float).reshape(-1)
    if len(pred)!=h or not np.isfinite(pred).all() or (pred<=0).any():
        raise ValueError('Forecast has incorrect length, nonfinite, or nonpositive values')
    anchor=float(np.mean(np.asarray(y,dtype=float)[-12:]))
    if (pred > FORECAST_MAX_LEVEL_MULTIPLE*anchor).any() or (pred < FORECAST_MIN_LEVEL_MULTIPLE*anchor).any():
        raise ValueError('Implausible extrapolation outside predeclared level-multiple bounds')
    return pred

Candidate specifications: 24 (ETS: 6 | SARIMA: 15 )


## 3. Rolling-origin cross-validation and separate specification tuning per CBSA

The **primary** design evaluates 12-month windows after December 2015, December 2016,
and December 2017. To examine the horizon mismatch separately, choose 24-, 36-, or
48-month windows with `RL_STAGE1_SELECTION_HORIZON` and
`RL_ALLOW_ALTERNATE_SELECTION=1` in a separate project root. All windows end
by June 2019. No fold sees data beyond June 2019. Each
individual ETS and SARIMA specification must pass *all three validation
origins* to qualify. The winning specification is the lowest mean normalized
RMSE, with normalized MAE as the tie-breaker (mean absolute error divided by that fold's training mean employment).

**Computational cost:** `standard` evaluates 3 baselines + 6 ETS + 15 SARIMA
candidates × 3 folds for each region, plus one or more final fits. 

In [4]:
FORECAST_DATES = pd.date_range(TRAIN_END+pd.offsets.MonthBegin(1),FORECAST_END,freq='MS')
assert len(FORECAST_DATES)==54 and FORECAST_DATES[0]==pd.Timestamp('2019-07-01')
origin_locs=[]
for origin in VALIDATION_ORIGINS:
    train_len=len(pd.date_range('2010-01-01',origin,freq='MS'))
    if train_len+VALIDATION_MONTHS>len(expected_pre_dates):
        raise ValueError(f'Validation date {origin} overlaps post-cutoff period')
    origin_locs.append((origin,train_len))

def evaluate_cbsa(cbsa, y):
    cv_rows=[]; summary_rows=[]; failed=[]; selected=None; forecast_rows=[]
    for candidate in candidates:
        fold_metrics=[]
        for origin,ntrain in origin_locs:
            t0=time.perf_counter()
            try:
                pred=fit_forecast(y[:ntrain],VALIDATION_MONTHS,candidate)
                truth=y[ntrain:ntrain+VALIDATION_MONTHS]
                mae=float(np.abs(truth-pred).mean())
                rmse=float(np.sqrt(np.mean((truth-pred)**2)))
                nmae=mae/float(y[:ntrain].mean())
                nrmse=rmse/float(y[:ntrain].mean())
                if not np.isfinite(nmae):
                    raise ValueError('Invalid normalized MAE')
                cv_rows.append({'cbsa_code':cbsa,'model':candidate['family'],
                    'specification':candidate['spec'],'origin':origin.date().isoformat(),
                    'mae':mae,'normalized_mae':nmae,'rmse':rmse,'normalized_rmse':nrmse,
                    'run_seconds':round(time.perf_counter()-t0,3)})
                fold_metrics.append((nrmse,nmae))
            except Exception as exc:
                failed.append({'cbsa_code':cbsa,'model':candidate['family'],
                    'specification':candidate['spec'],'origin':origin.date().isoformat(),
                    'error':f'{type(exc).__name__}: {exc}'})
        if len(fold_metrics)==len(origin_locs):
            score=float(np.mean([x[1] for x in fold_metrics]))
            rmse_score=float(np.mean([x[0] for x in fold_metrics]))
            summary_rows.append({'cbsa_code':cbsa,'model':candidate['family'],
                'specification':candidate['spec'],'cv_normalized_mae':score,'cv_normalized_rmse':rmse_score,
                'cv_folds':len(fold_metrics),'config_json':json.dumps(candidate['config'],sort_keys=True)})

    # Candidates failing any origin are ineligible. Resolve final fit failures
    # using only the training sample and previously calculated CV scores.
    ranked=sorted(summary_rows,key=lambda row:(row['cv_normalized_rmse'],row['cv_normalized_mae'],row['specification']))
    specs={candidate['spec']:candidate for candidate in candidates}
    for rank,row in enumerate(ranked, start=1):
        candidate=specs[row['specification']]
        try:
            p=fit_forecast(y,len(FORECAST_DATES),candidate)
            selected={'cbsa_code':cbsa,'model':candidate['family'],
                'specification':candidate['spec'],
                'config_json':json.dumps(candidate['config'],sort_keys=True),
                'cv_normalized_mae':row['cv_normalized_mae'],
                'cv_normalized_rmse':row['cv_normalized_rmse'],
                'selected_cv_rank':rank,'training_end':TRAIN_END.date().isoformat()}
            # These alternatives deliberately retain the *same* pre-cutoff input.
            reference=seasonal_naive(y,len(FORECAST_DATES))
            damping=np.exp(-np.arange(1,len(FORECAST_DATES)+1)/36.0)
            ensemble=0.5*p+0.5*reference
            damped=reference+damping*(p-reference)
            sensitivity={}
            for scenario,history,spec in (
                ('common_ets',y,{'family':'ets','spec':'common_damped_additive_ets','config':{'trend':'add','seasonal':'add','damped_trend':True}}),
                ('short_window',y[36:],candidate)):
                try:
                    sensitivity[scenario]=fit_forecast(history,len(FORECAST_DATES),spec)
                except Exception as exc:
                    sensitivity[scenario]=np.full(len(FORECAST_DATES),np.nan)
                    failed.append({'cbsa_code':cbsa,'model':spec['family'],'specification':spec['spec'],
                                   'origin':'sensitivity_'+scenario,'error':f'{type(exc).__name__}: {exc}'})
            forecast_rows=[{'cbsa_code':cbsa,'date':dt.strftime('%Y-%m-%d'),
                'counterfactual_employment':float(value),
                'counterfactual_ensemble_employment':float(ens),
                'counterfactual_damped_employment':float(damp),
                'counterfactual_common_ets_employment':float(common),
                'counterfactual_short_window_employment':float(short),
                'selected_model':candidate['family'],
                'selected_specification':candidate['spec']}
                for dt,value,ens,damp,common,short in zip(FORECAST_DATES,p,ensemble,damped,sensitivity['common_ets'],sensitivity['short_window'])]
            break
        except Exception as exc:
            failed.append({'cbsa_code':cbsa,'model':candidate['family'],
                'specification':candidate['spec'],'origin':'final',
                'error':f'{type(exc).__name__}: {exc}'})
    return selected,cv_rows,summary_rows,forecast_rows,failed

start=time.perf_counter()
if N_JOBS==1:
    results=[evaluate_cbsa(cbsa,y) for cbsa,y in tasks]
else:
    results=Parallel(n_jobs=N_JOBS,backend='loky',verbose=5)(
        delayed(evaluate_cbsa)(cbsa,y) for cbsa,y in tasks)

selected=pd.DataFrame([r[0] for r in results if r[0] is not None],
    columns=['cbsa_code','model','specification','config_json','cv_normalized_mae','cv_normalized_rmse',
             'selected_cv_rank','training_end'])
cv=pd.DataFrame([row for r in results for row in r[1]],
    columns=['cbsa_code','model','specification','origin','mae','normalized_mae','rmse','normalized_rmse','run_seconds'])
summary=pd.DataFrame([row for r in results for row in r[2]],
    columns=['cbsa_code','model','specification','cv_normalized_mae','cv_normalized_rmse','cv_folds','config_json'])
forecast=pd.DataFrame([row for r in results for row in r[3]],
    columns=['cbsa_code','date','counterfactual_employment','counterfactual_ensemble_employment',
             'counterfactual_damped_employment','counterfactual_common_ets_employment','counterfactual_short_window_employment','selected_model','selected_specification'])
failures=pd.DataFrame([row for r in results for row in r[4]],
    columns=['cbsa_code','model','specification','origin','error'])

# Always write audits, even if there were fit failures, so troubleshooting is possible.
cv.to_csv(OUT/'stage1_omb2018_cv_metrics.csv',index=False)
summary.to_csv(OUT/'stage1_omb2018_candidate_specification_scores.csv',index=False)
failures.to_csv(OUT/'stage1_omb2018_model_fit_failures.csv',index=False)
selected.to_csv(OUT/'stage1_omb2018_selected_models.csv',index=False)
if len(selected)!=len(tasks):
    missed=sorted(set(x for x,_ in tasks)-set(selected.cbsa_code))
    raise RuntimeError(f'No usable forecast for {len(missed)} CBSA(s); '
                       f'first missing IDs: {missed[:10]}. Inspect model_fit_failures.csv')
forecast.to_csv(PROC/'stage1_omb2018_forecasts_jul2019_dec2023.csv',index=False)
assert len(forecast)==len(tasks)*len(FORECAST_DATES)
assert not forecast.duplicated(['cbsa_code','date']).any()
print('Selected model families:')
print(selected.model.value_counts().to_string())
print('Unique SARIMA specifications chosen:',selected.loc[selected.model.eq('sarima'),'specification'].nunique())
print('Unique ETS configurations chosen:',selected.loc[selected.model.eq('ets'),'specification'].nunique())
print('Total elapsed minutes:',round((time.perf_counter()-start)/60,2))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    4.6s
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:   28.8s
[Parallel(n_jobs=8)]: Done 146 tasks      | elapsed:  1.2min
[Parallel(n_jobs=8)]: Done 272 tasks      | elapsed:  2.1min
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:  3.4min
[Parallel(n_jobs=8)]: Done 632 tasks      | elapsed:  4.9min
[Parallel(n_jobs=8)]: Done 866 tasks      | elapsed:  6.7min
[Parallel(n_jobs=8)]: Done 936 out of 936 | elapsed:  7.2min finished


Selected model families:
model
ets               420
sarima            293
trend_seasonal     94
seasonal_drift     78
seasonal_naive     51
Unique SARIMA specifications chosen: 15
Unique ETS configurations chosen: 6
Total elapsed minutes: 7.2


## 4. Hold-out diagnostic: July 2019–February 2020 (never used for fitting)

These eight later observed pre-pandemic months are *diagnostic only*.
Their measured values were not publicly eligible under the June 2019 QCEW
cutoff; they are not used to alter any selected model or hyperparameters.

In [5]:
forecast['date']=pd.to_datetime(forecast.date)
forecast_ids=set(forecast.cbsa_code)
future=panel.loc[panel.cbsa_code.isin(forecast_ids) &
                 panel.date.ge(pd.Timestamp('2020-03-01')),
                 ['cbsa_code','date','employment','cbsa_type']]
gaps=forecast.loc[forecast.date.ge(pd.Timestamp('2020-03-01'))].merge(
    future,on=['cbsa_code','date'],how='left',validate='one_to_one')
if gaps.employment.isna().any() or gaps.counterfactual_employment.isna().any():
    raise ValueError('Missing observed or predicted pandemic-period employment')
if not gaps.groupby('cbsa_code').size().eq(46).all():
    raise ValueError('Expected 46 months (Mar 2020–Dec 2023) for each CBSA')
gaps['gap_pct']=100*(gaps.employment/gaps.counterfactual_employment-1)
gaps['shortfall_pct']=(-gaps.gap_pct).clip(lower=0)
for alternative in ('ensemble','damped','common_ets','short_window'):
    col=f'counterfactual_{alternative}_employment'
    gaps[f'gap_pct_{alternative}']=100*(gaps.employment/gaps[col]-1)
    gaps[f'shortfall_pct_{alternative}']=(-gaps[f'gap_pct_{alternative}']).clip(lower=0)
GAPS_FILE=PROC/'stage1_omb2018_observed_vs_counterfactual_2020_2023.csv'
gaps.to_csv(GAPS_FILE,index=False)

cal=forecast.loc[forecast.date.le(pd.Timestamp('2020-02-01'))].merge(
    panel.loc[panel.cbsa_code.isin(forecast_ids),['cbsa_code','date','employment']],
    on=['cbsa_code','date'],how='left',validate='one_to_one')
if len(cal)!=len(tasks)*8 or cal.employment.isna().any():
    raise ValueError('Expected eight pre-pandemic diagnostic months per CBSA')
cal['gap_pct']=100*(cal.employment/cal.counterfactual_employment-1)
cal.to_csv(OUT/'stage1_omb2018_precovid_forecast_diagnostic.csv',index=False)

agg=gaps.groupby('date',as_index=False)[['employment','counterfactual_employment']].sum()
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(agg.date,agg.employment,label='Observed covered employment')
ax.plot(agg.date,agg.counterfactual_employment,'--',label='No-pandemic extrapolation')
ax.set(title='2018 OMB CBSAs: observed employment vs counterfactual',ylabel='Covered employment')
ax.legend();fig.tight_layout()
fig.savefig(OUT/'stage1_omb2018_observed_vs_forecast.png',dpi=200)
plt.close(fig)
print('Saved Stage 1 observed-vs-counterfactual:',len(gaps),'rows')
# Independent pre-pandemic July 2019--February 2020 forecast diagnostic.
cal['absolute_pct_error']=cal.gap_pct.abs()
cal['pred_minus_actual_pct']=100*(cal.counterfactual_employment/cal.employment-1)
precheck=cal.groupby('cbsa_code').agg(
    mean_gap_pct=('gap_pct','mean'),mean_absolute_pct_gap=('absolute_pct_error','mean'),
    mean_pred_minus_actual_pct=('pred_minus_actual_pct','mean')).reset_index()
precheck.to_csv(OUT/'stage1_omb2018_precovid_diagnostic_by_cbsa.csv',index=False)
print('Pre-pandemic diagnostic, July 2019--February 2020 (not used to choose models):')
print('Region-average gap bias (pp):',round(float(precheck.mean_gap_pct.mean()),3))
print('Region-average absolute gap (pp):',round(float(precheck.mean_absolute_pct_gap.mean()),3))
print('Median region-average absolute gap (pp):',round(float(precheck.mean_absolute_pct_gap.median()),3))
print('Largest diagnostic errors:')
print(precheck.nlargest(10,'mean_absolute_pct_gap').to_string(index=False))
# Note: all figures are diagnostics using observations not eligible for forecasting.


Saved Stage 1 observed-vs-counterfactual: 43056 rows
Pre-pandemic diagnostic, July 2019--February 2020 (not used to choose models):
Region-average gap bias (pp): -0.024
Region-average absolute gap (pp): 1.187
Median region-average absolute gap (pp): 0.85
Largest diagnostic errors:
cbsa_code  mean_gap_pct  mean_absolute_pct_gap  mean_pred_minus_actual_pct
    15700      7.774849               7.774849                   -7.023658
    21120     -7.255843               7.255843                    7.909941
    11640      7.175488               7.175488                   -6.685290
    42500     -7.100490               7.100490                    7.649145
    44900     -7.004299               7.004299                    7.575424
    21140     -6.578273               6.578273                    7.070166
    18220     -6.209901               6.209901                    6.628872
    42180      4.194648               5.787576                   -3.761960
    44780     -5.378780               5.378

## 5. Optional 54-month historical evaluation and illustrative counterfactual stress paths

Set `RL_STAGE1_LONG_CV=1` to generate historical pseudo-origin forecasts ending **no later than June 2019** (54 months from June 2013, June 2014, or December 2014). Within each pseudo-origin, choose among three predeclared baseline families using a **12-month validation block that ends at that origin**, then evaluate the next 12/24/36/48/54 months. The selection step never observes the future 54-month test block. These windows overlap and represent only three time environments; the results are not independent or calibrated probability intervals.

Each historical path records the ratio `historical actual / pre-origin forecast`. Applying its months 9–54 to the actual July-2019–December-2023 counterfactual creates three **illustrative sensitivity paths** starting March 2020. Log ratios are capped at a factor of 2 in either direction, with the cap rate audited. The base Stage 1 point estimates are unchanged. Notebook 03 exports the resulting alternative 12/24/46-month targets. Notebook 04 can rerun nested geographic validation on those targets with `RL_STAGE2_STRESS_EVAL=1`.

`RL_STAGE1_RETRO_SPEC=1` additionally estimates the finally selected 2019 model at older pseudo-origins for **descriptive model-family comparisons only**. That specification was selected using later data and must **not** be treated as a genuine historical out-of-time selection result.

Run the long historical evaluation for the full intended sample, not just a small set of regions, before using sensitivity results in the manuscript.


In [6]:
STRESS_MONTHLY_FILE=PROC/'stage1_omb2018_historical_stress_monthly.csv'
STRESS_OUTCOMES_FILE=PROC/'stage1_omb2018_historical_stress_outcomes.csv'
HISTORIC_LONG_FILE=OUT/'stage1_omb2018_long_horizon_backtests.csv'
HISTORIC_MONTHLY_FILE=OUT/'stage1_omb2018_historical_backtest_monthly.csv'
STRESS_ORIGINS=(pd.Timestamp('2013-06-01'),pd.Timestamp('2014-06-01'),
                pd.Timestamp('2014-12-01'))
HISTORIC_HORIZON=54
if LONG_CV:
    lookup={candidate['spec']:candidate for candidate in candidates}
    selected_by_id=selected.set_index('cbsa_code')
    def long_backtest_one(cbsa,y):
        score_rows=[]; monthly_rows=[]
        final_candidate=lookup[selected_by_id.loc[cbsa,'specification']]
        for origin in STRESS_ORIGINS:
            n=len(pd.date_range('2010-01-01',origin,freq='MS'))
            truth=y[n:n+HISTORIC_HORIZON]
            if len(truth)!=HISTORIC_HORIZON:
                raise ValueError('Historical test window extends beyond June 2019')
            # The prior 12-month block is never used in the fitting data of
            # the *baseline-selection* folds, and none of the 54 future months
            # enters either selection or refitting.
            selector_train=y[:n-12]; selector_holdout=y[n-12:n]
            ranking=[]
            for c in BASELINES:
                try:
                    forecast_12=fit_forecast(selector_train,12,c)
                    ranking.append((float(np.mean(np.abs(selector_holdout-forecast_12)) /
                                          np.mean(selector_train)),c['spec'],c))
                except Exception:
                    continue
            ranked=sorted(ranking,key=lambda r:(r[0],r[1]))
            chosen=None; historic_prediction=None; selection_mae=np.nan
            for score,_,c in ranked:
                try:
                    historic_prediction=fit_forecast(y[:n],HISTORIC_HORIZON,c)
                    chosen=c;selection_mae=score;break
                except Exception:
                    continue
            if chosen is None:
                # Robust, pre-origin-only fallback. This avoids aborting the national
                # stress analysis when ALL three seasonal benchmarks are unsuitable.
                # It is not pooled with the original Stage 1 candidate competition.
                fallback_level=float(np.mean(y[n-12:n]))
                if not np.isfinite(fallback_level) or fallback_level<=0:
                    raise RuntimeError(f'Invalid historical fallback level: {cbsa}, {origin.date()}')
                chosen={'family':'flat_last12_fallback','spec':'flat_last12_fallback','config':{}}
                historic_prediction=np.full(HISTORIC_HORIZON,fallback_level,dtype=float)
                selection_mae=np.nan
                score_rows.append({'cbsa_code':cbsa,'origin':str(origin.date()),
                    'strategy':'baseline_fallback_notice','specification':chosen['spec'],
                    'horizon_months':HISTORIC_HORIZON,'nmae':np.nan,
                    'error':'No baseline was admissible at the historical pseudo-origin; flat last-12-month mean used.'})
            run_models=[('preorigin_selected_baseline',chosen,historic_prediction,selection_mae)]
            if RETRO_SPEC:
                try:
                    retro_prediction=fit_forecast(y[:n],HISTORIC_HORIZON,final_candidate)
                    run_models.append(('retrospective_2019_selected',final_candidate,retro_prediction,np.nan))
                except Exception as exc:
                    score_rows.append({'cbsa_code':cbsa,'origin':str(origin.date()),
                        'strategy':'retrospective_2019_selected','specification':final_candidate['spec'],
                        'horizon_months':HISTORIC_HORIZON,'nmae':np.nan,
                        'error':f'{type(exc).__name__}: {exc}'})
            for strategy,c,pred,score in run_models:
                for horizon in (12,24,36,48,54):
                    score_rows.append({'cbsa_code':cbsa,'origin':str(origin.date()),
                        'strategy':strategy,'specification':c['spec'],
                        'family':c['family'],'horizon_months':horizon,
                        'nmae':float(np.mean(np.abs(truth[:horizon]-pred[:horizon]))/np.mean(y[:n])),
                        'mean_pred_minus_actual_pct':float(np.mean(100*(pred[:horizon]/truth[:horizon]-1))),
                        'selection_12month_nmae':score,'error':None})
                if strategy=='preorigin_selected_baseline':
                    for j,(actual,estimate) in enumerate(zip(truth,pred),1):
                        monthly_rows.append({'cbsa_code':cbsa,'origin':origin.strftime('%Y_%m'),
                            'horizon_month':j,'actual_historical':float(actual),
                            'forecast_historical':float(estimate),
                            'log_actual_over_forecast':float(np.log(actual/estimate)),
                            'selected_historical_specification':c['spec']})
        return score_rows,monthly_rows
    histories=(Parallel(n_jobs=N_JOBS,backend='loky',verbose=5)(
        delayed(long_backtest_one)(cbsa,y) for cbsa,y in tasks) if N_JOBS>1
        else [long_backtest_one(cbsa,y) for cbsa,y in tasks])
    historic_cv=pd.DataFrame([r for scores,_ in histories for r in scores])
    historical_monthly=pd.DataFrame([r for _,rows in histories for r in rows])
    historic_cv.to_csv(HISTORIC_LONG_FILE,index=False)
    historical_monthly.to_csv(HISTORIC_MONTHLY_FILE,index=False)
    print('Historical 54-month error diagnostics, with prior-origin baseline selection:')
    print(historic_cv.groupby(['strategy','horizon_months']).agg(
        median_nmae=('nmae','median'),median_pct_bias=('mean_pred_minus_actual_pct','median'),
        successful_fits=('nmae','count')).to_string())
    # Match the 2020-03 through 2023-12 target to historical horizons 9..54.
    # The common pseudo-origin preserves some shared historical shocks across CBSAs.
    historic_target=historical_monthly.loc[
        historical_monthly.horizon_month.between(9,54),
        ['cbsa_code','origin','horizon_month','log_actual_over_forecast']].copy()
    historic_target['date']=pd.Timestamp('2020-03-01')+pd.to_timedelta(0,unit='D')
    dates=pd.date_range('2020-03-01','2023-12-01',freq='MS')
    historic_target['date']=historic_target.horizon_month.map(
        {j:date for j,date in zip(range(9,55),dates)})
    observed=panel.loc[panel.cbsa_code.isin(selected.cbsa_code)&panel.date.ge('2020-03-01'),
                       ['cbsa_code','date','employment']]
    current=forecast.loc[forecast.date.ge('2020-03-01'),
                         ['cbsa_code','date','counterfactual_employment']]
    stress=historic_target.merge(observed,on=['cbsa_code','date'],validate='many_to_one')
    stress=stress.merge(current,on=['cbsa_code','date'],validate='many_to_one')
    if stress.groupby(['cbsa_code','origin']).size().ne(46).any():
        raise ValueError('Missing historical stress paths or target months')
    log_cap=np.log(2.0)
    stress['stress_log_ratio_capped']=stress.log_actual_over_forecast.clip(-log_cap,log_cap)
    stress['stress_cap_applied']=(stress.stress_log_ratio_capped!=stress.log_actual_over_forecast)
    stress['counterfactual_stress_employment']=(
        stress.counterfactual_employment*np.exp(stress.stress_log_ratio_capped))
    stress['gap_pct_stress']=100*(stress.employment/stress.counterfactual_stress_employment-1)
    stress['shortfall_pct_stress']=(-stress.gap_pct_stress).clip(lower=0)
    stress=stress.sort_values(['cbsa_code','origin','date'])
    stress.to_csv(STRESS_MONTHLY_FILE,index=False)
    scenario_rows=[]
    for (cbsa,origin),part in stress.groupby(['cbsa_code','origin']):
        scenario_rows.append({'cbsa_code':cbsa,'stress_origin':origin,
            'cumulative_shortfall_12_pp_months':float(part.shortfall_pct_stress.iloc[:12].sum()),
            'cumulative_shortfall_24_pp_months':float(part.shortfall_pct_stress.iloc[:24].sum()),
            'cumulative_shortfall_46_pp_months':float(part.shortfall_pct_stress.sum()),
            'signed_cumulative_gap_46_pp_months':float(-part.gap_pct_stress.sum()),
            'n_capped_months':int(part.stress_cap_applied.sum())})
    stress_outcomes=pd.DataFrame(scenario_rows)
    stress_outcomes.to_csv(STRESS_OUTCOMES_FILE,index=False)
    print('Illustrative historical stress paths:',stress_outcomes.stress_origin.nunique(),
          '| CBSAs:',stress_outcomes.cbsa_code.nunique(),
          '| capped ratio share:',round(float(stress.stress_cap_applied.mean()),4))
    print('Do NOT interpret these three historical stress paths as confidence intervals.')
else:
    print('Long-horizon historical stress tests NOT RUN. Set RL_STAGE1_LONG_CV=1 and rerun.')


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 144 tasks      | elapsed:    0.8s
[Parallel(n_jobs=8)]: Done 504 tasks      | elapsed:    3.0s
[Parallel(n_jobs=8)]: Done 936 out of 936 | elapsed:    5.4s finished


Historical 54-month error diagnostics, with prior-origin baseline selection:
                                            median_nmae  median_pct_bias  successful_fits
strategy                    horizon_months                                               
preorigin_selected_baseline 12                 0.013010        -0.480357             2808
                            24                 0.017546        -0.651109             2808
                            36                 0.022044        -0.704298             2808
                            48                 0.025427        -0.749771             2808
                            54                 0.027288        -0.773197             2808
Illustrative historical stress paths: 3 | CBSAs: 936 | capped ratio share: 0.0006
Do NOT interpret these three historical stress paths as confidence intervals.


## 6. Reproducibility manifest and handoff to Stage 2

In [7]:
manifest={
    'design_schema':'released_by_20200229_v2','selection_metric':'mean_normalized_rmse_then_mae','geography_vintage':'2018-09',
    'input_panel_sha256':sha256(PANEL),
    'information_cutoff':'2020-02-29',
    'model_training_cutoff':'2019-06-01',
    'rolling_validation_origins':[x.date().isoformat() for x in VALIDATION_ORIGINS],
    'validation_horizon_months':VALIDATION_MONTHS,
    'is_alternate_selection_design':VALIDATION_MONTHS!=12,
    'forecast_window':'2019-07 through 2023-12',
    'outcome_window':'2020-03 through 2023-12',
    'model_candidates':sorted(set(x['family'] for x in candidates)),
    'long_horizon_backtest_completed':LONG_CV,
    'historical_stress_paths_completed':LONG_CV,
    'historical_stress_origins':[x.strftime('%Y_%m') for x in STRESS_ORIGINS] if LONG_CV else [],
    'historical_stress_outcomes_sha256':sha256(STRESS_OUTCOMES_FILE) if LONG_CV else None,
    'historical_stress_monthly_sha256':sha256(STRESS_MONTHLY_FILE) if LONG_CV else None,
    'historical_backtest_monthly_sha256':sha256(HISTORIC_MONTHLY_FILE) if LONG_CV else None,
    'historical_retro_selected_spec_evaluation':RETRO_SPEC if LONG_CV else False,
    'historical_stress_interpretation':'Illustrative, 3 overlapping origin paths; not calibrated uncertainty intervals',
    'long_horizon_backtest_sha256':sha256(OUT/'stage1_omb2018_long_horizon_backtests.csv') if LONG_CV else None,
    'log_mean_bias_adjustment_enabled':LOG_BIAS_ADJUST,
    'forecast_upper_multiple':FORECAST_MAX_LEVEL_MULTIPLE,
    'forecast_lower_multiple':FORECAST_MIN_LEVEL_MULTIPLE,
    'counterfactual_sensitivities':['ensemble','damped','common_ets','short_window'],
    'prepandemic_diagnostic_sha256':sha256(OUT/'stage1_omb2018_precovid_diagnostic_by_cbsa.csv'),
    'candidate_specification_count':len(candidates),
    'ets_grid':ETS_GRID if RUN_ETS else 'disabled',
    'sarima_grid':SARIMA_GRID if RUN_SARIMA else 'disabled',
    'sarima_maxiter':SARIMA_MAXITER,
    'cbsa_count':len(tasks),
    'all_regions_included':MAX_CBSAS is None,
    'selected_model_count':len(selected),
    'total_validation_rows':len(cv),
    'total_failed_fit_attempts':len(failures),
    'selected_model_families':selected.model.value_counts().to_dict(),
    'historical_release_vintage_verified':False,
    'note':'QCEW historical downloads may contain revisions after the real-time information cutoff. Model selection is CBSA-specific, and candidate-level cross-validation scores are not independent accuracy estimates.',
    'selected_models_sha256':sha256(OUT/'stage1_omb2018_selected_models.csv'),
    'candidate_scores_sha256':sha256(OUT/'stage1_omb2018_candidate_specification_scores.csv'),
    'outcomes_sha256':sha256(GAPS_FILE),
    'counterfactual_forecasts_sha256':sha256(PROC/'stage1_omb2018_forecasts_jul2019_dec2023.csv'),
    'completed_utc':datetime.now(timezone.utc).isoformat(),
    'python_version':platform.python_version(),
    'pandas_version':pd.__version__,
}
(OUT/'stage1_omb2018_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Stage 1 complete.')

Stage 1 complete.
